# FMAPI → Databricks `ai_parse_document` 2.0

This notebook builds and deploys an MLflow pyfunc model that calls Databricks Foundation Model APIs for document parsing, then serves it behind a Model Serving endpoint. PDF pages are rasterized locally and sent as base64 images to the vision model endpoint. Update `CATALOG`, `SCHEMA`, and the `VISION_ENDPOINT` / `REFORMAT_ENDPOINT` names in the configuration cell below to point at your own Unity Catalog location and preferred FMAPI model before running.

Both model calls route through Databricks Foundation Model APIs:

1. **Vision endpoint** receives each rendered page image and extracts OCR text spans with bounding boxes and region captions.
2. **Reformat endpoint** groups and classifies OCR spans into the `ai_parse_document` 2.0 element taxonomy.
3. **Deterministic finalization** replaces model-controlled IDs/pages/metadata with authoritative values and validates the complete `ai_parse_document` 2.0 Pydantic envelope. If the LLM is unavailable, a schema-valid spans-only fallback is returned.

The endpoint output is a JSON string in a `response` column.

In [0]:
# No heavy ML packages needed — the vision model is called remotely via
# Databricks Foundation Model APIs. Only install orchestration and schema libraries.
%pip install -U \
  "mlflow>=3.6.0" \
  "pypdfium2>=4.30.0" \
  "openai>=1.37.0,<2.0.0"
%restart_python

In [0]:
# --- Workspace configuration ---
# Update CATALOG and SCHEMA to match your workspace's Unity Catalog location.
# Every cell below that registers the model or reads sample documents from a
# volume reuses these two values, so the notebook can be pointed at any
# workspace by changing them here only.
CATALOG = "justinm_demo"
SCHEMA = "parse"

# If this deployment is on GovCloud, you might need to set the AWS S3 Bucket in order to register the model in MLflow
# import os
# os.environ["AWS_DEFAULT_REGION"] = "us-gov-west-1"

In [0]:
import base64
import io
import json
import mimetypes
import time
import uuid
from pathlib import Path
from typing import Literal

import mlflow
import openai
import pandas as pd
from databricks.sdk import WorkspaceClient
from databricks.sdk.errors import NotFound
from mlflow import MlflowClient
from mlflow.models import ModelSignature
from mlflow.models.resources import DatabricksServingEndpoint
from mlflow.types.schema import ColSpec, DataType, Schema
from pydantic import BaseModel, ConfigDict, Field, field_validator, model_validator

mlflow.set_registry_uri("databricks-uc")

# Databricks Foundation Model API serving endpoints.
VISION_ENDPOINT = "databricks-gemini-3-8-flash"
REFORMAT_ENDPOINT = "databricks-gemini-3-8-flash"

REGISTERED_MODEL_NAME = f"{CATALOG}.{SCHEMA}.ai-parse-document-fmapi"
ENDPOINT_NAME = "ai-parse-document-fmapi"

# Sample documents used only to build the model signature below; update these
# paths to point at files that exist in your own Unity Catalog volume.
SAMPLE_FILES = [
    f"/Volumes/{CATALOG}/{SCHEMA}/docs/Cynthia-data-1-10-30-2024 (1).pdf",
    f"/Volumes/{CATALOG}/{SCHEMA}/docs/20260618 - Meal on plane.png",
]

RENDER_DPI = 200
DEFAULT_PAGE_LIMIT = 4
VISION_MAX_TOKENS = 4096
DSPY_MAX_TOKENS = 8192
DSPY_MAX_RETRIES = 3

print({
    "vision_endpoint": VISION_ENDPOINT,
    "registered_model": REGISTERED_MODEL_NAME,
    "endpoint": ENDPOINT_NAME,
    "reformat_endpoint": REFORMAT_ENDPOINT,
})

In [0]:
# Exact ai_parse_document schema version 2.0. `extra="forbid"` prevents
# syntactically valid replies from silently introducing non-contract fields.
ElementType = Literal[
    "text", "table", "figure", "title", "caption", "section_header",
    "page_header", "page_footer", "page_number", "footnote",
]

class StrictModel(BaseModel):
    model_config = ConfigDict(extra="forbid")

class Page(StrictModel):
    id: int = Field(ge=0)
    image_uri: str

class BoundingBox(StrictModel):
    coord: tuple[float, float, float, float]
    page_id: int = Field(ge=0)

    @field_validator("coord")
    @classmethod
    def ordered_box(cls, value):
        x0, y0, x1, y1 = value
        if x0 > x1 or y0 > y1:
            raise ValueError("bbox must satisfy x_min <= x_max and y_min <= y_max")
        return value

class Element(StrictModel):
    id: int = Field(ge=0)
    type: ElementType
    content: str | None
    confidence: float = Field(ge=0.0, le=1.0)
    bbox: list[BoundingBox] = Field(min_length=1)
    description: str | None

    @model_validator(mode="after")
    def validate_type_contract(self):
        if self.type == "figure":
            if self.content is not None or not self.description:
                raise ValueError("figure requires null content and a non-empty description")
        else:
            if not self.content or self.description is not None:
                raise ValueError("non-figure requires content and a null description")
            if self.type == "table" and not (
                "<table" in self.content.lower() and "</table>" in self.content.lower()
            ):
                raise ValueError("table content must be complete HTML table markup")
        return self

class Document(StrictModel):
    pages: list[Page]
    elements: list[Element]

class FileMetadata(StrictModel):
    file_path: str
    file_name: str
    file_size: int = Field(ge=0)
    file_modification_time: str

class Metadata(StrictModel):
    id: str
    version: Literal["2.0"]
    file_metadata: FileMetadata

class AiParseDocumentEnvelope(StrictModel):
    document: Document
    error_status: list[str]
    metadata: Metadata


In [0]:
PDF_MAGIC = b"%PDF"

def decode_b64(value: str) -> bytes:
    if not value:
        raise ValueError("file_b64 is required")
    value = value.strip()
    if value.startswith("data:"):
        value = value.split(",", 1)[1]
    return base64.b64decode(value, validate=False)

def decode_pages(file_b64, dpi=RENDER_DPI, page_limit=DEFAULT_PAGE_LIMIT, page_index=None):
    raw = decode_b64(file_b64)
    if raw[:4] == PDF_MAGIC:
        import pypdfium2 as pdfium
        pdf = pdfium.PdfDocument(raw)
        try:
            if page_index is not None:
                indices = [] if page_index >= len(pdf) else [page_index]
            else:
                indices = range(min(len(pdf), page_limit))
            result = []
            for page_id in indices:
                page = pdf[page_id]
                try:
                    image = page.render(scale=dpi / 72.0).to_pil().convert("RGB")
                    result.append((page_id, image))
                finally:
                    page.close()
            return result
        finally:
            pdf.close()
    from PIL import Image
    if page_index is not None and page_index > 0:
        return []
    return [(0, Image.open(io.BytesIO(raw)).convert("RGB"))]

def guess_mime_type(path: str) -> str:
    value = mimetypes.guess_type(path)[0]
    if value not in {"application/pdf", "image/png", "image/jpeg"}:
        raise ValueError(f"Unsupported file type: {path}")
    return value

def make_payload(path: str, page_limit=DEFAULT_PAGE_LIMIT, page_index=-1, reformat=True):
    stat = Path(path).stat()
    return {
        "file_b64": base64.b64encode(Path(path).read_bytes()).decode(),
        "mime_type": guess_mime_type(path),
        "file_path": path,
        "file_name": Path(path).name,
        "file_size": stat.st_size,
        "file_modification_time": pd.Timestamp(stat.st_mtime, unit="s", tz="UTC").isoformat(),
        "page_limit": page_limit,
        "page_index": page_index,
        "reformat": reformat,
    }

In [0]:
class AiParseDocumentModel(mlflow.pyfunc.PythonModel):
    """Vision OCR + typed reformatting + Pydantic validation."""

    # Prompt that instructs the vision model to produce structured OCR output with bboxes.
    _OCR_PROMPT = """You are a document OCR engine. Analyze this page image and extract all text
with precise bounding boxes. Return ONLY a JSON object with this exact structure:
{
  "spans": [{"text": "<verbatim text>", "bbox": [x_min, y_min, x_max, y_max]}],
  "region_captions": [{"caption": "<description of visual region>", "bbox": [x_min, y_min, x_max, y_max]}]
}

Rules:
- Bounding box coordinates must be normalized to a [0, 1000] scale relative to the image dimensions.
- Preserve all text verbatim — do not correct spelling, translate, or summarize.
- Order spans in reading order (top-to-bottom, left-to-right).
- For figures, charts, or non-text visual regions, add a region_caption describing the content.
- For tables, include the full table text as spans.
- Return valid JSON only, no markdown fences or explanation."""

    def __init__(self, vision_endpoint=VISION_ENDPOINT,
                 reformat_endpoint=REFORMAT_ENDPOINT, render_dpi=RENDER_DPI,
                 default_page_limit=DEFAULT_PAGE_LIMIT,
                 vision_max_tokens=VISION_MAX_TOKENS,
                 dspy_max_tokens=DSPY_MAX_TOKENS, dspy_max_retries=DSPY_MAX_RETRIES):
        self.vision_endpoint = vision_endpoint
        self.reformat_endpoint = reformat_endpoint
        self.render_dpi = render_dpi
        self.default_page_limit = default_page_limit
        self.vision_max_tokens = vision_max_tokens
        self.dspy_max_tokens = dspy_max_tokens
        self.dspy_max_retries = dspy_max_retries

    def load_context(self, context):
        import logging
        import openai
        from databricks.sdk import WorkspaceClient

        self.log = logging.getLogger("ai_parse_document")

        # WorkspaceClient uses the scoped serving credential created by MLflow's
        # DatabricksServingEndpoint resource declaration. Keep it around so a
        # fresh token can be pulled on every predict() call below — the scoped
        # credential is a short-lived OAuth token that can expire while a
        # served replica stays warm, and load_context() only runs once per
        # container lifecycle.
        self._ws = WorkspaceClient()
        serving_base = f"{self._ws.config.host}/serving-endpoints"

        self.vision_client = openai.OpenAI(
            api_key=self._get_token(),
            base_url=serving_base,
        )

        # Reformatting uses the same AI Gateway client and endpoint as OCR vision.
        # The Pydantic _finalize step still enforces the exact 2.0 schema.

    def _get_token(self):
        """Fetch a fresh bearer token (works across PAT, OAuth, serverless auth)."""
        token = self._ws.config.token
        if not token:
            headers = self._ws.config.authenticate()
            token = headers.get("Authorization", "").removeprefix("Bearer ").strip()
        return token

    def _extract_page(self, page_id, image):
        """Send a page image to the vision model and parse the structured OCR response."""
        import io as _io
        width, height = image.size

        # Encode image as PNG base64 for the vision API.
        buf = _io.BytesIO()
        image.save(buf, format="PNG")
        img_b64 = base64.b64encode(buf.getvalue()).decode()

        # Call the vision model via the OpenAI-compatible chat completions endpoint.
        response = self.vision_client.chat.completions.create(
            model=self.vision_endpoint,
            messages=[{
                "role": "user",
                "content": [
                    {"type": "text", "text": self._OCR_PROMPT},
                    {"type": "image_url", "image_url": {
                        "url": f"data:image/png;base64,{img_b64}"
                    }},
                ],
            }],
            max_tokens=self.vision_max_tokens,
        )

        content = response.choices[0].message.content
        # Handle both string and list-of-parts content formats.
        if isinstance(content, list):
            raw_text = "".join(
                part.get("text", "") if isinstance(part, dict) else getattr(part, "text", "")
                for part in content
            ).strip()
        else:
            raw_text = (content or "").strip()
        # Strip markdown code fences if the model wraps the JSON.
        if raw_text.startswith("```"):
            raw_text = raw_text.split("\n", 1)[1] if "\n" in raw_text else raw_text[3:]
            if raw_text.endswith("```"):
                raw_text = raw_text[:-3].strip()

        try:
            parsed = json.loads(raw_text)
        except json.JSONDecodeError as exc:
            self.log.warning("Vision model returned non-JSON for page %s: %s — attempting regex fallback", page_id, exc)
            # Regex fallback: extract individual spans even when overall JSON is broken.
            # Handles malformed bbox dicts, mixed "bbox"/"box" keys, missing brackets.
            import re
            span_pattern = re.compile(
                r'\{"text"\s*:\s*"((?:[^"\\]|\\.)*)"\s*,\s*"(?:bbox|box)"\s*:\s*'
                r'\[(\d+(?:\.\d+)?)\s*,\s*(\d+(?:\.\d+)?)\s*,\s*(\d+(?:\.\d+)?)\s*,\s*(\d+(?:\.\d+)?)\s*\]'
            )
            fallback_spans = []
            for m in span_pattern.finditer(raw_text):
                text = m.group(1).replace('\\n', '\n').replace('\\"', '"')
                coords = [float(m.group(i)) for i in range(2, 6)]
                if text.strip():
                    fallback_spans.append({"text": text, "bbox": coords})
            caption_pattern = re.compile(
                r'\{"caption"\s*:\s*"((?:[^"\\]|\\.)*)"\s*,\s*"(?:bbox|box)"\s*:\s*'
                r'\[(\d+(?:\.\d+)?)\s*,\s*(\d+(?:\.\d+)?)\s*,\s*(\d+(?:\.\d+)?)\s*,\s*(\d+(?:\.\d+)?)\s*\]'
            )
            fallback_captions = []
            for m in caption_pattern.finditer(raw_text):
                cap = m.group(1).replace('\\n', '\n').replace('\\"', '"')
                coords = [float(m.group(i)) for i in range(2, 6)]
                if cap.strip():
                    fallback_captions.append({"caption": cap, "bbox": coords})
            self.log.info("Regex fallback recovered %d spans, %d captions", len(fallback_spans), len(fallback_captions))
            parsed = {"spans": fallback_spans, "region_captions": fallback_captions}

        def _parse_bbox(bbox_raw):
            """Extract [x_min, y_min, x_max, y_max] from list or dict format."""
            if isinstance(bbox_raw, dict):
                # The model may return {"xmin": .., "ymin": .., "xmax": .., "ymax": ..}
                # or x_min/y_min/x_max/y_max variants.
                def _get(keys):
                    for k in keys:
                        if k in bbox_raw:
                            return float(bbox_raw[k])
                    return None
                vals = [
                    _get(["xmin", "x_min", "left", "x0", "x"]),
                    _get(["ymin", "y_min", "top", "y0", "y"]),
                    _get(["xmax", "x_max", "right", "x1", "width"]),
                    _get(["ymax", "y_max", "bottom", "y1", "height"]),
                ]
                if any(v is None for v in vals):
                    return None
                return vals
            elif isinstance(bbox_raw, (list, tuple)) and len(bbox_raw) == 4:
                try:
                    return [float(v) for v in bbox_raw]
                except (ValueError, TypeError):
                    return None
            return None

        # --- Phase 1: collect raw 0-1000 coords from the vision model ---------------
        spans_raw = []  # [{"text": str, "raw": [a, b, c, d]}]
        for span in parsed.get("spans") or []:
            text = str(span.get("text", "")).strip()
            coords = _parse_bbox(span.get("bbox") or span.get("box"))
            if text and coords:
                a, b, c, d = coords
                spans_raw.append({"text": text, "raw": [
                    min(a, c), min(b, d), max(a, c), max(b, d),
                ]})

        captions_raw = []  # [{"caption": str, "raw": [a, b, c, d]}]
        for rc in parsed.get("region_captions") or []:
            caption = str(rc.get("caption", "")).strip()
            coords = _parse_bbox(rc.get("bbox") or rc.get("box"))
            if caption and coords:
                a, b, c, d = coords
                captions_raw.append({"caption": caption, "raw": [
                    min(a, c), min(b, d), max(a, c), max(b, d),
                ]})

        # --- Phase 2: detect the model's axis order ---------------------------
        # The model nondeterministically returns either [x,y,x,y] (as prompted)
        # or [y,x,y,x]. We detect which by
        # checking whether the majority of spans are landscape (wider than
        # tall) — true for most document text. Operates in raw 0-1000 space.
        all_raw = [s["raw"] for s in spans_raw] + [c["raw"] for c in captions_raw]
        if all_raw:
            landscape_std = sum(1 for r in all_raw if (r[2] - r[0]) >= (r[3] - r[1]))
            landscape_swp = sum(1 for r in all_raw if (r[3] - r[1]) >= (r[2] - r[0]))
            axes_swapped = landscape_swp > landscape_std
            if axes_swapped:
                # Model returned [y_min, x_min, y_max, x_max]; reorder to
                # [x_min, y_min, x_max, y_max].
                for item in spans_raw + captions_raw:
                    y0, x0, y1, x1 = item["raw"]
                    item["raw"] = [x0, y0, x1, y1]

        # --- Phase 2b: per-span outlier correction -------------------------
        # After global axis detection, individual spans may still have
        # inconsistent axis ordering.  The vision model occasionally returns
        # the first span in [y,x,y,x] while the rest are [x,y,x,y].  A span
        # whose height exceeds 5× its width in the normalised 0-1000 space
        # is almost certainly axis-swapped — correct it individually.
        for item in spans_raw + captions_raw:
            x0, y0, x1, y1 = item["raw"]
            w, h = x1 - x0, y1 - y0
            if w > 0 and h > w * 5:
                item["raw"] = [y0, x0, y1, x1]

        # --- Phase 3: rescale 0-1000 → rendered-image pixels ---------------
        def _rescale(raw):
            return [
                round(raw[0] / 1000.0 * width, 1),
                round(raw[1] / 1000.0 * height, 1),
                round(raw[2] / 1000.0 * width, 1),
                round(raw[3] / 1000.0 * height, 1),
            ]

        spans = [{"text": s["text"], "bbox": _rescale(s["raw"])} for s in spans_raw]

        # Sort in reading order.
        band = max(height * 0.05, 1.0)
        spans.sort(key=lambda s: (int(((s["bbox"][1] + s["bbox"][3]) / 2.0) // band), s["bbox"][0]))

        region_captions = [
            {"caption": c["caption"], "bbox": _rescale(c["raw"])}
            for c in captions_raw
        ]

        return {"page_id": page_id, "width": width, "height": height,
                "spans": spans, "region_captions": region_captions}

    @staticmethod
    def _metadata(row, raw_size):
        # Optional MLflow columns can arrive as None or pandas NaN.
        def clean(name, default):
            value = row.get(name)
            return default if value is None or pd.isna(value) else value

        return Metadata(
            id=str(uuid.uuid4()),
            version="2.0",
            file_metadata=FileMetadata(
                file_path=str(clean("file_path", "uploaded_document")),
                file_name=str(clean("file_name", "uploaded_document")),
                file_size=int(clean("file_size", raw_size)),
                file_modification_time=str(
                    clean("file_modification_time", "1970-01-01T00:00:00Z")
                ),
            ),
        )

    def _fallback(self, pages, metadata, error=None):
        elements = []
        for page in pages:
            for span in page["spans"]:
                elements.append(Element(
                    id=len(elements), type="text", content=span["text"], confidence=0.5,
                    bbox=[BoundingBox(coord=tuple(span["bbox"]), page_id=page["page_id"])],
                    description=None,
                ))
        return AiParseDocumentEnvelope(
            document=Document(
                pages=[Page(id=p["page_id"], image_uri="uploaded_file") for p in pages],
                elements=elements,
            ),
            error_status=[] if error is None else [f"reformat_failed: {error}"],
            metadata=metadata,
        )

    def _finalize(self, candidate, pages, metadata):
        # Normalize the reformatter's output to match the strict 2.0 schema.
        # The model may use page_id instead of id, omit image_uri, use
        # different metadata fields, etc. We extract only elements
        # and rebuild the envelope structure ourselves.
        valid_page_ids = {p["page_id"] for p in pages}

        # Extract elements from candidate regardless of format.
        raw_doc = candidate if isinstance(candidate, dict) else candidate.dict()
        raw_elements = (raw_doc.get("document") or raw_doc).get("elements") or []

        elements = []
        for item in raw_elements:
            if isinstance(item, dict):
                el_type = item.get("type", "text")
                el_content = item.get("content")
                el_desc = item.get("description")
                el_conf = float(item.get("confidence", 0.5))
                # Normalize bboxes: accept list-of-dicts or list-of-lists.
                raw_boxes = item.get("bbox") or item.get("bounding_box") or []
                boxes = []
                for box in raw_boxes:
                    if isinstance(box, dict):
                        coord = box.get("coord") or box.get("coords") or box.get("coordinates")
                        page_id = box.get("page_id", 0)
                    elif isinstance(box, (list, tuple)) and len(box) == 4:
                        coord = box
                        page_id = 0
                    else:
                        continue
                    if coord and len(coord) == 4 and page_id in valid_page_ids:
                        boxes.append(BoundingBox(
                            coord=tuple(float(c) for c in coord), page_id=page_id
                        ))
                if not boxes:
                    continue
                description = el_desc if el_type == "figure" else None
                content = None if el_type == "figure" else el_content
                if content is None and description is None:
                    continue
                elements.append(Element(
                    id=len(elements), type=el_type, content=content,
                    confidence=max(0.0, min(1.0, el_conf)),
                    bbox=boxes, description=description,
                ))
            else:
                # Already a Pydantic model (shouldn't happen with dict path).
                pass

        if any(page["spans"] for page in pages) and not elements:
            raise ValueError("Reformatter returned no usable elements for non-empty OCR")
        return AiParseDocumentEnvelope(
            document=Document(
                pages=[Page(id=p["page_id"], image_uri="uploaded_file") for p in pages],
                elements=elements,
            ),
            error_status=[],
            metadata=metadata,
        )

    _REFORMAT_SYSTEM = """You are a document structure parser. Convert the raw OCR spans and region
captions into a Databricks ai_parse_document version 2.0 JSON envelope.

Preserve OCR text verbatim: do not correct, paraphrase, translate, summarize, or invent
content. Group adjacent spans into reading-order blocks. Use only these element types:
text, table, figure, title, caption, section_header, page_header, page_footer,
page_number, footnote. A table's content must be valid HTML. A figure's content must be
null and its description must be grounded only in region captions. Compute each element
bbox as the exact union of its source span boxes; coordinates are rendered-image pixels.
Never merge across pages. Confidence is in [0,1]. Return every required field.

Return ONLY a valid JSON object with this exact top-level structure:
{"document": {"pages": [...], "elements": [...]}, "error_status": [], "metadata": {...}}

Each element: {"id": int, "type": str, "content": str|null, "confidence": float,
"bbox": [{"coord": [x_min, y_min, x_max, y_max], "page_id": int}], "description": str|null}

Return valid JSON only, no markdown fences or explanation."""

    @staticmethod
    def _extract_text_from_content(content):
        """Extract plain text from OpenAI-compatible content (handles thinking blocks)."""
        if isinstance(content, str):
            return content.strip()
        if isinstance(content, list):
            parts = []
            for part in content:
                if isinstance(part, dict) and part.get("type") == "text":
                    parts.append(part.get("text", ""))
                elif hasattr(part, "text") and getattr(part, "type", None) == "text":
                    parts.append(part.text)
            return "\n".join(parts).strip()
        return str(content or "").strip()

    def _reformat(self, pages, metadata):
        payload = json.dumps({"pages": pages}, ensure_ascii=False)
        last_error = None
        for attempt in range(self.dspy_max_retries):
            try:
                response = self.vision_client.chat.completions.create(
                    model=self.reformat_endpoint,
                    messages=[
                        {"role": "system", "content": self._REFORMAT_SYSTEM},
                        {"role": "user", "content": f"OCR output:\n{payload}"},
                    ],
                    max_tokens=self.dspy_max_tokens,
                )
                raw_text = self._extract_text_from_content(
                    response.choices[0].message.content
                )
                # Strip markdown fences if present.
                if raw_text.startswith("```"):
                    raw_text = raw_text.split("\n", 1)[1] if "\n" in raw_text else raw_text[3:]
                    if raw_text.endswith("```"):
                        raw_text = raw_text[:-3].strip()
                candidate = json.loads(raw_text)
                return self._finalize(candidate, pages, metadata)
            except Exception as exc:
                last_error = exc
                self.log.warning("Reformat attempt %s failed: %s", attempt + 1, exc)
                if attempt + 1 < self.dspy_max_retries:
                    time.sleep(2 ** attempt)
        return self._fallback(pages, metadata, last_error)

    def _handle_row(self, row):
        raw = decode_b64(row.get("file_b64"))
        raw_index = row.get("page_index")
        page_index = None
        if raw_index is not None and not pd.isna(raw_index) and int(raw_index) >= 0:
            page_index = int(raw_index)
        raw_limit = row.get("page_limit")
        page_limit = (
            self.default_page_limit
            if raw_limit is None or pd.isna(raw_limit)
            else int(raw_limit)
        )
        decoded = decode_pages(
            row["file_b64"], dpi=self.render_dpi,
            page_limit=page_limit,
            page_index=page_index,
        )
        metadata = self._metadata(row, len(raw))
        pages = [self._extract_page(page_id, image) for page_id, image in decoded]
        reformat = row.get("reformat")
        if reformat is not None and not pd.isna(reformat) and not bool(reformat):
            return json.dumps({"pages": pages}, ensure_ascii=False)
        envelope = self._fallback(pages, metadata) if not pages else self._reformat(pages, metadata)
        # model_dump_json is the final schema gate and stable JSON serializer.
        return envelope.model_dump_json()

    def predict(self, context, model_input, params=None):
        # Refresh the cached client's api_key on every scoring call. The
        # openai SDK reads self.api_key live when building request headers
        # (auth_headers is a property, not baked in at __init__), so this is
        # enough to pick up a renewed token without rebuilding the client.
        self.vision_client.api_key = self._get_token()
        if isinstance(model_input, pd.DataFrame):
            rows = model_input.to_dict(orient="records")
        elif isinstance(model_input, dict):
            rows = [model_input]
        else:
            rows = list(model_input)
        output = []
        for row in rows:
            try:
                output.append(self._handle_row(row))
            except Exception as exc:
                self.log.exception("Row failed")
                metadata = self._metadata(row if isinstance(row, dict) else {}, 0)
                output.append(self._fallback([], metadata, f"row_error: {exc}").model_dump_json())
        return pd.DataFrame({"response": output})

In [0]:
# Build a minimal single-page PDF in memory so the input_example doesn't
# depend on any files existing in the workspace.
_MINIMAL_PDF = (
    b"%PDF-1.4\n"
    b"1 0 obj<</Type/Catalog/Pages 2 0 R>>endobj\n"
    b"2 0 obj<</Type/Pages/Kids[3 0 R]/Count 1>>endobj\n"
    b"3 0 obj<</Type/Page/MediaBox[0 0 612 792]/Parent 2 0 R>>endobj\n"
    b"xref\n0 4\n"
    b"0000000000 65535 f \n"
    b"0000000009 00000 n \n"
    b"0000000058 00000 n \n"
    b"0000000115 00000 n \n"
    b"trailer<</Size 4/Root 1 0 R>>\nstartxref\n190\n%%EOF"
)

# Log and register. Both endpoint resources give the served pyfunc scoped
# credentials for vision OCR calls and reformatting.
signature = ModelSignature(
    inputs=Schema([
        ColSpec(DataType.string, "file_b64", required=True),
        ColSpec(DataType.string, "mime_type", required=False),
        ColSpec(DataType.string, "file_path", required=False),
        ColSpec(DataType.string, "file_name", required=False),
        ColSpec(DataType.long, "file_size", required=False),
        ColSpec(DataType.string, "file_modification_time", required=False),
        ColSpec(DataType.long, "page_limit", required=False),
        ColSpec(DataType.long, "page_index", required=False),
        ColSpec(DataType.boolean, "reformat", required=False),
    ]),
    outputs=Schema([ColSpec(DataType.string, "response", required=True)]),
)

input_example = pd.DataFrame([{
    "file_b64": base64.b64encode(_MINIMAL_PDF).decode(),
    "mime_type": "application/pdf",
    "file_path": "sample.pdf",
    "file_name": "sample.pdf",
    "file_size": len(_MINIMAL_PDF),
    "file_modification_time": "1970-01-01T00:00:00Z",
    "page_limit": 1,
    "page_index": -1,
    "reformat": True,
}])

with mlflow.start_run(run_name="fmapi_ai_parse_document") as run:
    model_info = mlflow.pyfunc.log_model(
        name="model",
        python_model=AiParseDocumentModel(),
        signature=signature,
        input_example=input_example,
        resources=[
            DatabricksServingEndpoint(endpoint_name=VISION_ENDPOINT),
            DatabricksServingEndpoint(endpoint_name=REFORMAT_ENDPOINT),
        ],
        pip_requirements=[
            "--extra-index-url=https://pypi.org/simple",
            "mlflow>=3.6.0",
            "pillow>=10.4.0",
            "pypdfium2>=4.30.0",
            "openai>=1.37.0,<2.0.0",
            "pandas>=2.2.0",
            "numpy>=1.26.0,<2.3.0",
            "databricks-sdk>=0.62.0",
            "pydantic==2.10.6",
            "cloudpickle==3.1.2",
        ],
        metadata={
            "vision_endpoint": VISION_ENDPOINT,
            "reformat_endpoint": REFORMAT_ENDPOINT,
            "output_schema": "ai_parse_document/2.0",
        },
        # registered_model_name omitted to avoid metastore quota check;
        # register the version manually below.
    )
    logged_run_id = run.info.run_id
    logged_model_uri = model_info.model_uri

# Register a new version; create the registered model first if it doesn't exist.
client = MlflowClient(registry_uri="databricks-uc")

try:
    client.get_registered_model(REGISTERED_MODEL_NAME)
except Exception:
    client.create_registered_model(REGISTERED_MODEL_NAME)
new_version = client.create_model_version(
    name=REGISTERED_MODEL_NAME,
    source=logged_model_uri,
    run_id=logged_run_id,
)
latest_model_version = new_version.version
client.set_registered_model_alias(
    name=REGISTERED_MODEL_NAME, alias="Production", version=latest_model_version
)
version_uri = f"models:/{REGISTERED_MODEL_NAME}/{latest_model_version}"
print({"run_id": logged_run_id, "model_uri": version_uri, "alias": "Production"})

In [0]:
# Create or update a CPU serving endpoint. No GPU needed — all inference
# is done via remote FMAPI calls.
from databricks.sdk.service.serving import (
    EndpointCoreConfigInput,
    ServedEntityInput,
)

workspace = WorkspaceClient()
served_entity = ServedEntityInput(
    name=ENDPOINT_NAME,
    entity_name=REGISTERED_MODEL_NAME,
    entity_version=latest_model_version,
    workload_size="Small",
    scale_to_zero_enabled=False,
)

try:
    workspace.serving_endpoints.get(name=ENDPOINT_NAME)
    workspace.serving_endpoints.update_config(
        name=ENDPOINT_NAME, served_entities=[served_entity]
    )
    action = "updated"
except NotFound:
    workspace.serving_endpoints.create(
        name=ENDPOINT_NAME,
        config=EndpointCoreConfigInput(served_entities=[served_entity]),
    )
    action = "created"

# Deployment is asynchronous; poll until ready or fail explicitly.
deadline = time.time() + 60 * 60
while time.time() < deadline:
    endpoint = workspace.serving_endpoints.get(name=ENDPOINT_NAME)
    ready = str(getattr(endpoint.state, "ready", None))
    updating = str(getattr(endpoint.state, "config_update", None))
    print({"ready": ready, "config_update": updating})
    if "READY" in ready and ("NOT_UPDATING" in updating or updating in {"None", "READY"}):
        break
    if "FAILED" in updating:
        raise RuntimeError(f"Endpoint deployment failed: {endpoint.state}")
    time.sleep(30)
else:
    raise TimeoutError(f"Timed out waiting for {ENDPOINT_NAME}")

print({"action": action, "endpoint": ENDPOINT_NAME, "version": latest_model_version})

In [0]:
# ---------- Smoke test: call the endpoint via requests ----------
import requests as _requests

ws = WorkspaceClient()
_token = ws.config.token
if not _token:
    _token = ws.config.authenticate().get("Authorization", "").removeprefix("Bearer ").strip()
_host = ws.config.host

# Build payload from the first sample file (single page, with reformat)
SAMPLE_FILE = "/Volumes/justinm_demo/bio_track/unstructured/Cynthia-data-1-10-30-2024.pdf"
payload_row = make_payload(SAMPLE_FILE, page_limit=1, reformat=True)

resp = _requests.post(
    f"{_host}/serving-endpoints/{ENDPOINT_NAME}/invocations",
    headers={"Authorization": f"Bearer {_token}", "Content-Type": "application/json"},
    json={"dataframe_records": [payload_row]},
    timeout=300,
)
print(f"Status: {resp.status_code}")
if resp.status_code == 200:
    result = resp.json()
    doc_json = result["predictions"][0]["response"]
    doc = json.loads(doc_json)
    print(f"Pages:    {len(doc['document']['pages'])}")
    print(f"Elements: {len(doc['document']['elements'])}")
    print(f"Errors:   {doc['error_status']}")
    print("---")
    for el in doc["document"]["elements"][:5]:
        text = (el.get("content") or el.get("description") or "")[:120]
        print(f"  [{el['type']}] {text}")
else:
    print(resp.text[:500])

In [0]:
%sql
-- Smoke test: call the endpoint via ai_query in SQL.
-- Reads a sample PDF from the UC volume, base64-encodes it, and sends it
-- to the pyfunc endpoint. Returns the raw ai_parse_document 2.0 JSON.
SELECT ai_query(
    'ai-parse-document-fmapi',
    named_struct(
        'file_b64',                 base64(content),
        'mime_type',                'application/pdf',
        'file_path',                path,
        'file_name',                regexp_extract(path, '[^/]+$', 0),
        'file_size',                CAST(length AS LONG),
        'file_modification_time',   CAST(modificationTime AS STRING),
        'page_limit',               CAST(1 AS LONG),
        'page_index',               CAST(-1 AS LONG),
        'reformat',                 true
    )
) AS parsed_result
FROM read_files(
    '/Volumes/justinm_demo/bio_track/unstructured/Cynthia-data-1-10-30-2024.pdf',
    format => 'binaryFile'
)
LIMIT 1